In [ ]:
직접 구성한 CNN < resnet50 < efficientNetb1 < efficientNetb0 < efficientNetb4
66%             75%         83%                 85%             87%
데이터가 부족한 상황으로 생각하여 10개의 클래스에서 증강기법을 적용해서 작업을 진행하였으나
1. 유사 이미지의 등장
2. 같은 이미지의 촬영 위치가 다른 이미지
3. 걸러내지 못한 잘못된 이미지
4. 주관적 판단하에 이뤄진 직접적 데이터 전처리
이러한 과정들에 의해서 증강기법을 사용하는 것보다 있는 그대로의 모델을 사용하는 편이 훨씬 좋았음.
또한 스케일링도 keras에서 제공하는 스케일링보다 모델에 적합한 스케일링 및 풀링을 적용하는 것이 성능에 긍정적 영향을 미치는 것을 확인함
주로 epochs는 20내외에서 중단되며 adam을 위주로 사용하였고 adamw를 사용하거나 rmsprop을 사용했을 경우 
Adam보다 빠른 학습을 보이나 과적합(rmsprop), 손실값의 유지 등(adamw)의 문제를 확인하게 됨

해당 상황과 더불어 모델별 loss값을 확인했을 때에도 efficientNetB4가 학습에서 그나마 손실 안정성이 낮은 편이었으며,
다른 모델들에 비해 정확도도 제일 높은 모습을 보였음

In [ ]:
image_dataset_from_directory 해당 메서드는 이미지 확장자에 대한 전처리를 자동화해주므로 전처리를 간략하게 하기 위해 차용함
- 이미지 resize
- 확장자 자동 맞춤
- 폴더에 따른 클래스 분류
- 배치 사이즈 설정
- 이미지 그레이스케일로 지정
- PIL 라이브러리를 내부적으로 적용하고 있음
위의 이유로 전처리에 들이는 시간을 줄이게 됨
전처리는 

In [ ]:
# 새(종) 미세 분류기
# 1. 데이터를 정리
# 2. 파일을 ds 화 해야 한다
# 3. 데이터 증강 사용
# 4. 전이학습 사용할 것
# 5. 성능 점수는 자유
# https://www.kaggle.com/datasets/wenewone/cub2002011

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
print(f"Keras backend: {keras.backend.backend()}")

Keras backend: torch


In [2]:
from keras.applications import ResNet50, VGG16, efficientnet
from keras.activations import relu, leaky_relu, sigmoid, softmax
from keras.losses import binary_crossentropy, sparse_categorical_crossentropy, categorical_crossentropy
from keras.layers import Dense, Conv2D, GlobalAveragePooling2D, Input
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.datasets import cifar100
from keras.utils import image_dataset_from_directory

# GoogLeNet

In [3]:
from keras.datasets import cifar100
import numpy as np

(tr_x, tr_y), (tt_x, tt_y) = cifar100.load_data()

In [4]:
from keras import regularizers
from keras.layers import Activation, BatchNormalization, MaxPool2D, concatenate, Flatten, Input, Conv2D, Dense
from keras.models import Model
from keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

def cv2d_bn_l(x, filters, kernel_size, weight_decay=.0, strides=1):
    x = Conv2D(filters, kernel_size, strides, 'same', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization(scale=False, axis=3)(x)
    x = Activation('relu')(x)
    return x

def inception_module(x, fs_num_l, weight_decay=.0):
    br0_f, br1_f, br2_f, br3_f = fs_num_l
    br0 = cv2d_bn_l(x, br0_f, 1, weight_decay)

    br1 = cv2d_bn_l(x, br1_f[0], 1, weight_decay)
    br1 = cv2d_bn_l(br1, br1_f[1], 3, weight_decay)

    br2 = cv2d_bn_l(x, br2_f[0], 1, weight_decay)
    br2 = cv2d_bn_l(br2, br2_f[1], 5, weight_decay)

    br3 = MaxPool2D(pool_size=3, strides=1, padding='same')(x)
    br3 = cv2d_bn_l(br3, br3_f, 1, weight_decay)

    x = concatenate([br0, br1, br2, br3], axis=3)
    return x

In [5]:
def googlenet(input_shape, classes, weight_decay=.0):
    input_l = Input(shape=(input_shape))
    x = input_l
    x = cv2d_bn_l(x, 64, 1, weight_decay)
    x = cv2d_bn_l(x, 192, 3, weight_decay)
    x = MaxPool2D(3,2,'same')(x)
    x = inception_module(x, (64, (96, 128), (16,32), 32), weight_decay)
    x = inception_module(x, (128, (128, 192), (32,96), 64), weight_decay)
    x = MaxPool2D(3,2,'same')(x)
    x = inception_module(x, (192, (96, 208), (16,48), 64), weight_decay)
    x = inception_module(x, (160, (112, 224), (24,64), 64), weight_decay)
    x = inception_module(x, (128, (128, 256), (24,64), 32), weight_decay)
    x = inception_module(x, (112, (144, 288), (32,96), 64), weight_decay)
    x = inception_module(x, (256, (160, 320), (32,128), 128), weight_decay)
    x = MaxPool2D(2,2,'same')(x)
    x = inception_module(x, (256, (160, 320), (32,128), 128), weight_decay)
    x = inception_module(x, (384, (192, 384), (48,128), 128), weight_decay)
    x = Flatten()(x)
    output_l = Dense(classes, activation='softmax')(x)
    m = Model(input_l, output_l)
    return m

In [6]:
input_shape = (32,32)
c_n = 3
batch_size = 64
weight_decay = 5e-4
l_r = 1e-2 # 0.01
epoch = 10
class_n = 100
ggn_m = googlenet(input_shape+(3,), class_n, weight_decay)

from keras.losses import sparse_categorical_crossentropy
from keras.optimizers import SGD
op = SGD(learning_rate=l_r, momentum=0.9)
ggn_m.compile(optimizer=op, loss=sparse_categorical_crossentropy, metrics=['acc'])
reduce_lr = ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-7, verbose=1)
es = EarlyStopping(patience=10, restore_best_weights=True)
ck = ModelCheckpoint('b_m.keras', save_best_only=True)


hy = ggn_m.fit(tr_x, tr_y, batch_size=batch_size, epochs=epoch, validation_split=.2, callbacks=[reduce_lr, es, ck])

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step - acc: 0.0655 - loss: 8.8540

c:\Users\devchoi\miniconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\devchoi\miniconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\devchoi\miniconda3\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violat

625/625 ━━━━━━━━━━━━━━━━━━━━ 161s 253ms/step - acc: 0.0951 - loss: 8.4102 - val_acc: 0.1353 - val_loss: 9.1864 - learning_rate: 0.0100
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 123s 197ms/step - acc: 0.1983 - loss: 7.2068 - val_acc: 0.0522 - val_loss: 13.2207 - learning_rate: 0.0100
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 190ms/step - acc: 0.2526 - loss: 6.6052 - val_acc: 0.2284 - val_loss: 7.1165 - learning_rate: 0.0100
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 126s 201ms/step - acc: 0.2980 - loss: 6.0742 - val_acc: 0.2002 - val_loss: 7.5324 - learning_rate: 0.0100
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 126s 201ms/step - acc: 0.3505 - loss: 5.4931 - val_acc: 0.2815 - val_loss: 6.3964 - learning_rate: 0.0100
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 123s 196ms/step - acc: 0.4104 - loss: 4.9920 - val_acc: 0.3490 - val_loss: 4.9843 - learning_rate: 0.0100
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 125s 201ms/step - acc: 0.4599 - loss: 4.5312 - val_acc: 0.3475 - val_loss: 4.9334 - learning_r

In [ ]:
왜 이미지가 640?
그리고 왜 v8을 전처리에서 이용함?